# PycisTopic tutorial

This is introduction tutorial notebook for pycisTopic v3 package. It covers basic functionalities of the package including data import, pseudobulk profile creation, consensus peak inference, quality check, topic modelling and downstream analysis

In [15]:
!pycistopic -h

usage: pycistopic [-h] {qc,topic_modeling,tss,count_matrix} ...

pycisTopic CLI.

options:
  -h, --help            show this help message and exit

Commands:
  List of available commands for pycisTopic CLI.

  {qc,topic_modeling,tss,count_matrix}
                        Command description.
    qc                  Run QC statistics on fragment file.
    topic_modeling      Run LDA topic modeling.
    tss                 Get TSS gene annotation from Ensembl BioMart.
    count_matrix        Generate binary fragment count matrix for multiple
                        samples.


# Dowloading the data

In this tutorial, we will be using in-house generated PBMC data that can be downloaded from GEO.

In [ ]:
!mkdir -p data


# Output diretory set up

Let's create an ouptput directory

In [20]:
import os
out_dir = "outs"
os.makedirs(out_dir, exist_ok = True)

# Getting pseudobulk profiles from cell annotations

TODO

# Inferring consensus peaks

TODO

# Quality check 

The next step is to perform QC for the scATAC-seq samples (in this case, only one run). There are several measurements and visualizations performed in this step:
- Barcode rank plot
- Duplication rate
- Insertion size
- TSS enrichment
- Fraction of Reads In Peaks (FRIP)

To calculate the TSS enrichment we need to provide TSS annotations. You can easily download them via the pycistopic `tss get_tss command`.

In case you are unsure which column name is used by [Ensembl](https://www.ensembl.org/index.html) to specify gene names in their databases, run the `pycistopic tss gene_annotation_list` and grep for your species.

In [16]:
!pycistopic tss gene_annotation_list | grep Human

hsapiens_gene_ensembl	Human genes (GRCh38.p14)


In [17]:
!mkdir -p outs/qc
!pycistopic tss get_tss \
    --output outs/qc/tss.bed \
    --name "hsapiens_gene_ensembl" \
    --to-chrom-source ucsc \
    --ucsc hg38

- Get TSS annotation from Ensembl BioMart with the following settings:
  - biomart_name: "hsapiens_gene_ensembl"
  - biomart_host: "http://www.ensembl.org"
  - transcript_type: ['protein_coding']
  - use_cache: True
/lustre1/project/stg_00002/lcb/apancik/software/pycisTopic/.venv/lib/python3.14/site-packages/pybiomart/dataset.py:269: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  result = pd.read_csv(StringIO(response.text), sep='\t')
- Getting chromosome sizes and alias mapping for "hg38" from UCSC.
- Update chromosome names in TSS annotation to "ucsc" chromosome names.
- Writing TSS annotation BED file to "outs/qc/tss.bed".


In [18]:
!head outs/qc/tss.bed | column -t

#     Chromosome  Start  End      Gene  Score  Strand          Transcript_type  Ensembl_gene_id
chrM  3306        3307   MT-ND1   .     +      protein_coding  ENSG00000198888  
chrM  4469        4470   MT-ND2   .     +      protein_coding  ENSG00000198763  
chrM  5903        5904   MT-CO1   .     +      protein_coding  ENSG00000198804  
chrM  7585        7586   MT-CO2   .     +      protein_coding  ENSG00000198712  
chrM  8365        8366   MT-ATP8  .     +      protein_coding  ENSG00000228253  
chrM  8526        8527   MT-ATP6  .     +      protein_coding  ENSG00000198899  
chrM  9206        9207   MT-CO3   .     +      protein_coding  ENSG00000198938  
chrM  10058       10059  MT-ND3   .     +      protein_coding  ENSG00000198840  
chrM  10469       10470  MT-ND4L  .     +      protein_coding  ENSG00000212907  


Next, let’s calculate the QC metrics using the `pycistopic qc` command.

In [ ]:
!pycistopic qc \
    --fragments /fragments.tsv.gz \
    --regions outs/consensus_peak_calling/consensus_regions.bed \
    --tss outs/qc/tss.bed \
    --output outs/qc/pbmc

usage: pycistopic qc [-h] {run,filter,plot} ...
pycistopic qc: error: argument qc: invalid choice: 'data/fragments.tsv.gz' (choose from run, filter, plot)


In case you have multiple samples, you can run the QC step in parallel as follows.

In [33]:
fragments_dict = { 'B_cell' : 'fragments/B_cell.fragments.tsv.gz',
'CD14' : 'fragments/CD14+_monocyte.fragments.tsv.gz',
'CD16' : 'fragments/CD16+_monocyte.fragments.tsv.gz',
'CD4' : 'fragments/CD4+_T_cell.fragments.tsv.gz',
'Cyto' : 'fragments/Cytotoxic_T_cell.fragments.tsv.gz',
'Dendritic' : 'fragments/Dendritic_cell.fragments.tsv.gz',
'NK' : 'fragments/Natural_killer_cell.fragments.tsv.gz'
}

In [34]:
import os 
regions_bed_filename = os.path.join(out_dir, "consensus_regions.bed")
tss_bed_filename = os.path.join(out_dir, "qc", "tss.bed")

pycistopic_qc_commands_filename = "pycistopic_qc_commands.txt"

# Create text file with all pycistopic qc command lines.
with open(pycistopic_qc_commands_filename, "w") as fh:
    for sample, fragment_filename in fragments_dict.items():
        print(
            "pycistopic qc run",
            f"--fragments {fragment_filename}",
            f"--regions {regions_bed_filename}",
            f"--tss {tss_bed_filename}",
            f"--output {os.path.join(out_dir, "qc")}/{sample}",
            sep=" ",
            file=fh,
        )

And run the following in a command line environment.

In [ ]:
!cat pycistopic_qc_commands.txt | parallel -j 4 {}

Academic tradition requires you to cite works you base your article on.
If you use programs that use GNU Parallel to process data for an article in a
scientific publication, please cite:

  Tange, O. (2024, December 22). GNU Parallel 20241222 ('Bashar').
  Zenodo. https://doi.org/10.5281/zenodo.14550073

This helps funding further development; AND IT WON'T COST YOU A CENT.
If you pay 10000 EUR you should feel free to use GNU Parallel without citing.

More about funding GNU Parallel and the citation notice:
https://www.gnu.org/software/parallel/parallel_design.html#citation-notice

To silence this citation notice: run 'parallel --citation' once.

Come on: You have run parallel 15 times. Isn't it about time 
you run 'parallel --citation' once to silence the citation notice?



We can visualize sample level statistics.

These include:

- **Barcode rank plot**: The barcode rank plot shows the distribution of non-duplicate reads and which barcodes were inferred to be associated with cells. A steep drop-off (‘knee’) is indicative of good separation between the cell-associated barcodes and the barcodes associated with empty partitions.
- **Insertion size**: ATAC-seq requires a proper pair of Tn5 transposase cutting events at the ends of DNA. In the nucleosome-free open chromatin regions, many molecules of Tn5 can kick in and chop the DNA into small pieces; around nucleosome-occupied regions, and Tn5 can only access the linker regions. Therefore, in a good ATAC-seq library, you should expect to see a sharp peak at the <100 bp region (open chromatin), and a peak at ~200bp region (mono-nucleosome), and other larger peaks (multi-nucleosomes). A clear nucleosome pattern indicates a good quality of the experiment.
- **Sample TSS enrichment**: The TSS enrichment calculation is a signal to noise calculation. The reads around a reference set of TSSs are collected to form an aggregate distribution of reads centered on the TSSs and extending to 1000 bp in either direction (for a total of 2000bp). This distribution is then normalized by taking the average read depth in the 100 bps at each of the end flanks of the distribution (for a total of 200bp of averaged data) and calculating a fold change at each position over that average read depth. This means that the flanks should start at 1, and if there is high read signal at transcription start sites (highly open regions of the genome) there should be an increase in signal up to a peak in the middle.

We can also visualize barcode level statistics.

These statistics can be used to filter cell barcodes to retain only high quality cells.

The barcode level statistics include:

- **Total number of (unique) fragments**
- **TSS enrichment**: The score at position in the TSS enrichment score for for each barcode (at position 0, the TSS). Noisy cells will have a low TSS enrichment.
- **FRIP**: The fraction of reads in peaks for each barcode. Noisy cells have low FRIP values. However, this filter should be used with nuance, as it depends on the quality of the original peaks. For example, if there is a rare population in the sample, its specific peaks may be missed by peak calling algorithms, causing a decrease in their FRIP values.

In [ ]:
!pycistopic qc plot -s SAMPLE_ID -o outs/qc/pbmc

In [ ]:
# TODO not sure what the names of output files are
import matplotlib.pyplot as plt
image = plt.imread('outs/qc/pbmc/sample_stats.png')
plt.imshow(image)


<div class="alert alert-info">

**Note:**

The `pycistopic qc filter` command will determine automatic thresholds for the minimum number of unique number of fragments and the minumum TSS enrichment.
In case you want to change these thresholds or want to threhold based on FRIP, you can provide manually defined thresholds using the parameters:
- unique_fragments_threshold
- tss_enrichment_threshold
- frip_threshold
    
In this case we will use the automatically defined thresholds, **please manualy inspect the quality metrics to make sure these thresholds are valid!**
    
</div>


In [ ]:
!pycistopic qc filter -s SAMPLE_ID -o outs/qc/pbmc

# Count matrix

This step involves generating a count matrix containing fragment counts over consensus peaks (see above) for each cell barcode passing the QC metrices defined above.

Blacklist regions will be removed from this count matrix.

In [ ]:
! pycistopic count_matrix -h

In [ ]:
! sample_to_fragment_path=""
! sample_to_cell_bc_path=""
! region_ids_path="consensus_regions.bed"
! regions_out=""
! cb_out="" 
! matrix_out=""
! path_to_blacklist=""
pycistopic count_matrix -f "${sample_to_fragment_path}" -c "${sample_to_cell_bc_path}" -r "${region_ids_path}" --out_region_names "${regions_out}" 
                            --out_cell_barcodes "${cb_out}" --out_matrix "${matrix_out}"  -b "${path_to_blacklist}" -d -1 -s ___

# Topic modelling (Mallet)

Next we will perform the actual topic modeling using LDA using a Collapsed Gibbs Sampler.

There are two options for topic modeling, which both produce similar results:

- **Parallel LDA with MALLET (recommended):** The parallelization is done within each model. Recommended for large data sets where a few models with different number of topics are being tested. If working in a cluster, we recommed to submit a job per model so they can run simultaneously. 
Command: `pycistopic topic_modeling mallet `

- **Serial LDA:** The parallelization is done between models rather than within each model. Recommended for small-medium sized data sets in which several models with different number of topics are being tested. Command: `pycistopic topic_modeling lda`

Here, we will use Mallet. 

<div class="alert alert-info">

**Note:**

In order to be able to run Mallet you need the Mallet binary, these binaries can be downloaded from Github. You can also compile the binary from source, for more information please visit the Mallet Github repository.

<div/>

In [ ]:
!wget https://github.com/mimno/Mallet/releases/download/v202108/Mallet-202108-bin.tar.gz
!tar -xf Mallet-202108-bin.tar.gz

<div class="alert alert-info">

**Note:**

In order to run Mallet topic modeling, you need to have Java installed/loaded

<div/>

In [ ]:
## get corpus:
! module load Java/19.0.2
! binary_matrix_path="/binary_accessibility_matrix.mtx"
! corpus_out_path="/corpus"
! mallet_path="../software/mallet/Mallet-202108/bin/mallet"
! pycistopic topic_modeling mallet create_corpus -i "${binary_matrix_path}" -o "${corpus_out_path}" -m 100 -b "${mallet_path}" 

Because we don’t know yet what number of topics will be optimal for our dataset we will run several topic models, each with a different number of topics.

In [ ]:
## run mallet topic modeling:
! topics = 20 30 40 50 ## not sure if this will work
! corpus_path="/corpus"
! output_prefix="out/models/tm"
! mallet_path="../software/mallet/Mallet-202108/bin/mallet"
! pycistopic topic_modeling mallet run -i "${corpus_path}" -o "${output_prefix}" -t "${topics}" -p 8 -n 150 -a 50 -A True -e 0.1 -E False -s 555 -m 300 -b ."${mallet_path}" -v

<div class="alert alert-secondary">

**Note:**

Topic modeling may take some time, so you can also create a script per topic model to submit it in the background.

Example code:

```bash
topics=(20 30 40 50)
mallet_path="../software/mallet/Mallet-202108/bin/mallet"
scripts_dir="./scripts/topic_modeling/"
corpus_path="./corpus"
output_prefix="out/models/tm"
mkdir -p "$scripts_dir"
mkdir -p "$output_prefix"

for topic in "${topics[@]}"; do
    script="${scripts_dir}${topic}.sh"
    cat > "$script" <<EOL
#!/bin/bash -l
# Command for topic ${topic}
pycistopic topic_modeling mallet run -i $corpus_path -o $output_prefix -t $topic -p 8 -n 150 -a 50 -A True -e 0.1 -E False -s 555 -m 300 -b $mallet_path -v
EOL

    echo "Created script: $script"
done

In [ ]:
## Generate AnnData h5ad file from mallet result
! topics = 20 30 40 50 ## not sure if this will work
! cells_path="/cell_barcodes.txt"
! regions_path="/region_names.txt"
! output_prefix="out/models/topic_modeling_result_"
pycistopic topic_modeling mallet create_anndata -c "${cells_path}" -r "${regions_path}" -o "${output_prefix}" -t "${topics}"

**Model selection**

Let’s select the model with the optimal number of topics.

There is no optimal way to do this selection, however getting the exact optimal number of topics is also not critical.

To make the selection easier we implemented several metrics:

- **Minmo_2011:** Uses the average model coherence as calculated by Mimno et al (2011). In order to reduce the impact of the number of topics, we calculate the average coherence based on the top selected average values. The better the model, the higher coherence.

- **Log-likelihood:** Uses the log-likelihood in the last iteration as calculated by Griffiths and Steyvers (2004). The better the model, the higher the log-likelihood.

- **Arun_2010:** Uses a density-based metric as in Arun et al (2010) using the topic-region distribution, the cell-topic distribution and the cell coverage. The better the model, the lower the metric.

- **Cao_Juan_2009:** Uses a divergence-based metric as in Cao Juan et al (2009) using the topic-region distribution. The better the model, the lower the metric.

In [ ]:
## run topic model statistics:
! topics = 20 30 40 50 ## not sure if this will work
! binary_matrix_path="/binary_accessibility_matrix.mtx"
! cells_path="/cell_barcodes.txt"
! regions_path="/region_names.txt"
! output_prefix="out/models/tm"
! pycistopic topic_modeling mallet stats -i "${binary_matrix_path}" -c "${cells_path}" -r "${regions_path}" -o "${output_prefix}" -t "${topics}" -v

In [ ]:
## plot topic models statistics:
! output_prefix="out/models/tm"
! topics = 20 30 40 50 ## not sure if this will work
! pycistopic topic_modeling mallet plot_stats -o "${output_prefix} -t "${topics}" -q png

<div class="alert alert-info">

**Note:**

Note that for both the Arun and Cao metric a lower score is related to a better model, for visualization purposes we inverted these scores. In the plots below a higher score for these metrics are thus related to a better model.

Also note that not all metrics agree (see for example the Arun metric).


<div/>

For scATAC-seq data models, the most helpful methods are Minmo (topic coherence) and the log-likelihood in the last iteration.

In [ ]:
## paste png plot here

# Topic binarization & QC

After selecting the model, we can binarize topic-region and cell-topic distributions. The first is useful for exploring the topics with other tools that work with region sets (e.g. GREAT, cisTarget); while the latter is useful to automatically annotate topics.

We will first binarize the topic-region distributions. There are several methods that can be used for this: `otsu` (Otsu, 1979), `yen` (Yen et al., 1995), `li` (Li & Lee, 1993), `aucell` (Van de Sande et al., 2020) or `ntop` (Taking the top n regions per topic). Otsu and Yen’s methods work well for topic-region distributions; however for some downstream analyses (e.g. deep learning) it may be convenient to use ntop to have balanced region sets. Here, we will show `ntop` and `otsu`.

In [ ]:
## binarize topic-region probabilities using the top 3000 regions per topic
! topics = 40 ## replace with final selected
! cells_path="/cell_barcodes.txt"
! regions_path="/region_names.txt"
! outdir="out/models/binarized_regions/"
! output_prefix="binarized_topic_regions_ntop_"
pycistopic topic_modeling mallet binarize -a region -m ntop -n 3000 -s True -c "${cells_path}" -r "${regions_path}" -o "${output_prefix}" -t "${topics}" -p "${outdir}"

In [ ]:
## binarize topic-region probabilities using otsu_thresholding
! topics = 40 ## replace with final selected
! cells_path="/cell_barcodes.txt"
! regions_path="/region_names.txt"
! outdir="out/models/binarized_regions/"
! output_prefix="binarized_topic_regions_otsu_"
pycistopic topic_modeling mallet binarize -a region -m otsu -s True -b 100 -c "${cells_path}" -r "${regions_path}" -o "${output_prefix}" -t "${topics}" -p "${outdir}"

In [ ]:
## add histograms

Similarly, we can now binarize the cell-topic distribions.

In [ ]:
## binarize cell-topic probabilities
! topics = 40 ## replace with final selected
! cells_path="/cell_barcodes.txt"
! regions_path="/region_names.txt"
! outdir="out/models/binarized_cells/"
! output_prefix="binarized_cell_topic_ntop_"
pycistopic topic_modeling mallet binarize -a cell -m ntop -n 3000 -s True -c "${cells_path}" -r "${regions_path}" -o "${output_prefix}" -t "${topics}" -p "${outdir}"

In [ ]:
## binarize topic-region probabilities using otsu_thresholding
! topics = 40 ## replace with final selected
! cells_path="/cell_barcodes.txt"
! regions_path="/region_names.txt"
! outdir="out/models/binarized_cells/"
! output_prefix="binarized_cell_topic_otsu_"
pycistopic topic_modeling mallet binarize -a cell -m otsu -s True -b 100 -c "${cells_path}" -r "${regions_path}" -o "${output_prefix}" -t "${topics}" -p "${outdir}"

### Topic QC
Following, we can compute the topic quality control metrics. These include:

- Number of assignments

- Topic coherence (Mimno et al., 2011): Measures to which extent high scoring regions in the topic are actually co-accessible in the original data. If it is low it indicates that the topic is rather random. The higher, the better is a topic.

- The marginal topic distribution: Indicates how much each topic contributes to the model. The higher, the better is a topic.

- The gini index: Value between 0 and 1, that indicates the specificity of topics (0: General, 1:Specific)

If topics have been binarized, the number of regions/cells per topic will be added.



In [ ]:
## add topic QC commandline

Next, we can automatically annotate topics, in this case by cell type. Here we calculate the proportion of cells in each group that are assigned to the binarized topic in comparison to the ratio in the whole data set. We will consider a topic as general if the difference between the ration of cells in the whole data set in the binarized topic and the ratio of total cells in the assigned groups is above 0.2. This indicates that the topic is general, and the propotion test may fail if the topic is enriched in both foreground (the group) and background (the whole data set); resulting in a big difference between the ratios.



In [ ]:
## add topic annotation commandline

# DARs

to do

# Signature enrichment

to do

# Gene activity ?

# Clustering and plotting
- cell topic heatmap
- scanpy umap and tsne
- scanpy umap and tsne plots

